# MM4250: take data

One **take** = the switch's internal standards (ALL_OPEN, short, load), each chosen channel, then the standards again (the drift check), `repeats` times (from the cooldown file). Every position is saved raw (S11 and S21; all four with `SWEEP_MODE = "2port"`) to the lab database (`<project>_Cooldown_<NN>_vna.db`), then each channel is corrected with NIST's definitions (295 K or 3 K, picked from the fridge temperature) and saved as its own runs, linked to the raw ones.

- Run the cells top to bottom. Everything about the setup (ports, fridge lines, sweep) is in the cooldown file; this notebook only says which file.
- Only one notebook can hold the switch: run the other notebook's **Close** cell first.
- VNA port 1 on the input line, port 2 on the return line (see the file's `lines`). The LNA must be on, or nothing comes back.
- The switch is always left `ALL_OPEN` at the end of a take, even if it fails. See `README.md`.

**Setup**: which cooldown file. Prints what it says; fix the file if anything is wrong.

In [ ]:
COOLDOWN_FILE = "cooldowns/CD05.json"   # this cooldown's file (see README.md)

import sys
from pathlib import Path
try:
    here = Path(_dh[0])                      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent                       # the folder with drivers/ in it
sys.path[:0] = [str(here), str(root)]
%matplotlib inline

import vna, db, ecal, plots
from scripts import switch_vna as sv   # the lab's safe switch + VNA functions
cfg = db.load_cooldown(here / COOLDOWN_FILE)
db.print_cooldown(cfg)

**Connect**: VNA, switch, fridge (optional: without the VPN it warns and carries on, with no temperatures), and the database.

In [ ]:
from qcodes.instrument import Instrument
from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250

Instrument.close_all()                       # so re-running this cell is safe
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
ksvna.output(False)                          # source off first: connecting the switch moves it
switch = MM4250("switch")                    # connecting sets ALL_OPEN

fridge = None                                # only for temperatures: optional
try:
    from maybell_qcodes import MaybellFridge
    fridge = MaybellFridge("fridge", host="10.179.200.168")   # needs the GlobalConnect VPN
except Exception as e:
    print(f"[warning] no fridge ({type(e).__name__}: {e}); carrying on without temperatures")
print("Fridge (mixing plate):", sv.format_kelvin(sv.read_temperatures(fridge)["mxp"]))

DB = db.open_database(cfg, "vna")            # E:\\QCoDeS\\<project>\\Cooldown_<NN>\\..._vna on the DAQ

**Take**: about a minute per channel at 2001 points (longer with more). `POINTS` and `POWER_DBM` set the sweep (the band and IF bandwidth come from the cooldown file); every run records what the VNA was actually set to. `SWEEP_MODE` picks what each sweep measures: `"forward"` (default) is S11 and S21 with only VNA port 1 sourcing, so nothing is sent down the return line into the LNA's output; `"2port"` adds S12/S22 (port 2 sources too, at `POWER_DBM`). Everything here is analysed from S21 (S11 on the bench), and those are the same numbers in both modes. Forward also sets port 2's source Off on the VNA itself (it stays Off afterwards), and refuses to sweep while another trace (e.g. S22) is on the VNA: delete it there, or run `sv.remove_traces(ksvna)`. `CHANNELS = None` measures every port in the file; `[4]` just RF4 (still with both sets of standards). Prints the drift at the end: compare with NIST's few 1e-2.

In [ ]:
CHANNELS = None          # None = every port in the cooldown file; or e.g. [4]
POINTS = 2001            # sweep points across the file's band (e.g. 20001 = 5 kHz steps over 100 MHz)
POWER_DBM = -20          # VNA source power, dBm (at most 0); the fridge's input line attenuates it
SWEEP_MODE = "forward"   # "forward": port 1 only, S11 + S21, nothing into the LNA output (safe default);
                         # "2port": all four, port 2 also sources at POWER_DBM down the return line
NOTE = ""                # anything about this take: fridge state, why you took it
SAVE_TOUCHSTONE = True  # True: also .s2p/.s1p files in Sweeps/takes/ beside the database
DEFINITIONS = None       # None = from the fridge: 295K from 250 K up, 3K at 4.5 K and below,
                         # raw only in between or with no reading; or "295K" / "3K" by hand

save = db.Saver(DB, "take", files=SAVE_TOUCHSTONE)
t = vna.take(db.with_sweep(cfg, POINTS, POWER_DBM, SWEEP_MODE), ksvna, switch, fridge,
             channels=CHANNELS, note=NOTE, save=save)   # ends ALL_OPEN
take = db.load_take(t["take_id"], DB)
result = ecal.correct_take(take, DEFINITIONS)          # None: no temperature and no DEFINITIONS
if result is not None:
    db.save_corrected(DB, result, take, files=SAVE_TOUCHSTONE)
    take = db.load_take(t["take_id"], DB)
db.backup_db(DB)
if SAVE_TOUCHSTONE:
    print("Files in", db.take_folder(DB, take))
ecal.drift(take);

**Plot latest take**: run right after Take. Left: raw trace the correction uses; right: corrected at each connector. |S| in dB on top, phase below.

In [ ]:
SAVE_FIGURE = False      # True: take.png in the take's Sweeps/takes/ folder (never overwrites)

plots.plot_take(take, save=db.take_folder(DB, take) / "take.png" if SAVE_FIGURE else None);

**Recall**: any time, no instruments needed (only Setup has to have run). Lists every take in the database, then plots one (or a list of runs) and can export Touchstone files.

In [ ]:
RECALL_DB = db.open_database(cfg, "vna", create=False)   # this cooldown's; or a path, e.g. an earlier cooldown's
TAKE_ID = None           # None = the newest take; or one from the list, e.g. "20261108T141502.318"
RUN_IDS = None           # or a list of run ids instead of a take, e.g. [12, 13]
EXPORT = False           # True: Touchstone files in Sweeps/export/ beside that database (never overwrites)
CORRECT_WITH = None      # "295K" or "3K": correct a take that was saved raw only (no temperature then)

db.list_takes(RECALL_DB)
if RUN_IDS:
    plots.plot_runs(db.load_runs(RUN_IDS, RECALL_DB))
    ids = RUN_IDS
else:
    old = db.load_take(TAKE_ID, RECALL_DB)
    if CORRECT_WITH:
        db.save_corrected(RECALL_DB, ecal.correct_take(old, CORRECT_WITH), old)
        old = db.load_take(old["take_id"], RECALL_DB)
    plots.plot_take(old)
    ids = db.take_run_ids(old)
if EXPORT:
    db.export_touchstone(ids, RECALL_DB)

**Hold the thru line open**: connects the thru channel to RFC and leaves it, so you can use the VNA from its own software (e.g. to check the HEMT). Meanwhile, every `INTERVAL_S` (10 s) it reads every fridge thermometer (MXP, MXP2, STP, CMN, STP2, CFP, PRP): each reading goes into the CSV as it's taken (`SAVE_CSV`: the cooldown folder's `Sweeps/thru_hold/<time>_temps.csv`), and the table under this cell updates in place with the newest ones. A thermometer that can't be read shows `--` (empty in the CSV). Run the next cell when you're done.

Power dissipated at the switch while it's held:
- Gate (holding it closed): under 1 uW (Menlo datasheet)
- RF, source off: 0
- RF, source on: about 7 pW at -40 dBm from the VNA, about 0.7 nW at -20 dBm

In [ ]:
from datetime import datetime
INTERVAL_S = 10    # seconds between fridge readings
SAVE_CSV = True    # True: each reading appended to Sweeps/thru_hold/<time>_temps.csv as it's taken
SHOW_LAST = 6      # readings shown in the live table under this cell

if globals().get("hold") is not None:
    sv.release(switch, ksvna, hold)      # a hold from an earlier run of this cell
csv_path = (db.open_database(cfg, "vna", create=False).parent / "Sweeps" / "thru_hold"
            / f"{datetime.now():%Y%m%d_%H%M%S}_temps.csv")
ksvna.output(False)                      # leave the source off while it's held
hold = sv.hold_state(switch, ksvna, cfg["thru_channel"], fridge=fridge, interval_s=INTERVAL_S,
                     csv_path=csv_path, save_csv=SAVE_CSV, show_last=SHOW_LAST)

**Open it again**: back to ALL_OPEN, stops the fridge logger (after the reading in progress) and prints how many readings it took, the last one, and where the CSV is.

In [ ]:
sv.release(switch, ksvna, hold)          # ALL_OPEN first, then stops the log and prints a summary
hold = None

**Close**: before anyone else needs the switch or VNA. Backs up the database (skipped if nothing is new), leaves the switch `ALL_OPEN` and releases everything.

In [ ]:
db.backup_db(DB)
sv.all_open(switch, ksvna)                    # source off for the move
print("switch, last sent:", switch.state.cache.get(get_if_invalid=False))   # not read back: the lab driver's readback lags
ksvna.output(False)
ksvna.close()
switch.close()
if fridge is not None:
    fridge.close()